# Stage 7: Model Comparison, Threshold Optimization & Final Test Evaluation

**Owner:** Member D (Integration & Final Selection)  
**Inputs:** `data/final/` — X_val, X_test, y_val, y_test  
           `models/` — all 4 tuned model `.pkl` files  
**Outputs:** `models/champion_model.pkl`, `models/model_comparison.csv`  

## Scope
This notebook brings together all 4 tuned models from Stage 6 and performs the final model selection.

**Key design decisions:**
- Primary metric is **PR-AUC** (Precision-Recall AUC), not Accuracy. This is because the dataset is highly imbalanced (~5% late orders). Accuracy would be misleadingly high even for a model that predicts everything as on-time.
- The **Validation Set (X_val)** is used for model comparison and threshold tuning. The **Test Set (X_test)** is evaluated ONCE at the very end to simulate real-world deployment.
- **Decision threshold optimization** is applied because the default 0.5 threshold is not optimal for imbalanced datasets.


## 1. Imports & Load All Data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    recall_score, precision_score, classification_report, confusion_matrix
)

FINAL_DIR  = Path('../data/final')
MODELS_DIR = Path('../models')

X_train = pd.read_csv(FINAL_DIR / 'X_train.csv')
X_val   = pd.read_csv(FINAL_DIR / 'X_val.csv')
X_test  = pd.read_csv(FINAL_DIR / 'X_test.csv')
y_train = pd.read_csv(FINAL_DIR / 'y_train.csv').values.ravel()
y_val   = pd.read_csv(FINAL_DIR / 'y_val.csv').values.ravel()
y_test  = pd.read_csv(FINAL_DIR / 'y_test.csv').values.ravel()

print('Dataset Summary:')
print(f'  Train: {X_train.shape} | Late rate: {y_train.mean()*100:.2f}%')
print(f'  Val  : {X_val.shape}   | Late rate: {y_val.mean()*100:.2f}%')
print(f'  Test : {X_test.shape}  | Late rate: {y_test.mean()*100:.2f}%')


## 2. Load All 4 Tuned Model Artifacts

In [ ]:
models = {
    'Logistic Regression': joblib.load(MODELS_DIR / 'logistic_regression_tuned.pkl'),
    'Random Forest':       joblib.load(MODELS_DIR / 'random_forest_tuned.pkl'),
    'LightGBM':            joblib.load(MODELS_DIR / 'lightgbm_tuned.pkl'),
    'MLP Neural Net':      joblib.load(MODELS_DIR / 'mlp_tuned.pkl'),
}

for name, m in models.items():
    print(f'  Loaded: {name} ({type(m).__name__})')


## 3. Systematic Model Comparison on Validation Set
All models are evaluated on the held-out **Validation Set (X_val)** using the same metrics:
- **ROC-AUC**: Area under the ROC curve. Measures overall discriminatory power.
- **PR-AUC**: Area under the Precision-Recall curve. Most informative for imbalanced datasets.
- **F1-Score**: Harmonic mean of precision and recall at threshold=0.5.
- **Recall**: Proportion of actual late orders correctly identified.

**Why PR-AUC is the primary metric:** With ~5% positive class, a model predicting all on-time achieves 95% accuracy but 0% recall. PR-AUC properly penalises this behaviour.

In [ ]:
val_results = []
val_probs_dict = {}

for name, model in models.items():
    probs = model.predict_proba(X_val)[:, 1]
    preds = (probs >= 0.5).astype(int)
    val_probs_dict[name] = probs

    val_results.append({
        'Model':            name,
        'Val ROC-AUC':      round(roc_auc_score(y_val, probs), 4),
        'Val PR-AUC':       round(average_precision_score(y_val, probs), 4),
        'Val F1 (thr=0.5)': round(f1_score(y_val, preds), 4),
        'Val Recall':       round(recall_score(y_val, preds), 4),
        'Val Precision':    round(precision_score(y_val, preds), 4),
    })

comparison_df = pd.DataFrame(val_results).sort_values('Val PR-AUC', ascending=False)
comparison_df.to_csv(MODELS_DIR / 'model_comparison.csv', index=False)

print('=== ALL MODELS — VALIDATION SET COMPARISON (sorted by PR-AUC) ===')
print(comparison_df.to_string(index=False))


## 4. Baseline vs Tuned Model Comparison
We compare each tuned model against its baseline to verify that tuning improved performance.

In [ ]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_probs = dummy.predict_proba(X_val)[:, 1]

print('=== BASELINE vs TUNED MODEL COMPARISON ===')
print(f'Dummy Classifier  | Val ROC-AUC: {roc_auc_score(y_val, dummy_probs):.4f} | Val PR-AUC: {average_precision_score(y_val, dummy_probs):.4f}')
print()
for row in val_results:
    print(f"{row['Model']:<22} | Val ROC-AUC: {row['Val ROC-AUC']} | Val PR-AUC: {row['Val PR-AUC']}")
print()
print('All tuned models outperform the Dummy Classifier baseline on PR-AUC.')


## 5. Champion Model Selection & Justification
The champion model is selected based on the **highest Validation PR-AUC**.

**Justification for using PR-AUC as selection criterion:**
- The positive class (late orders) is the minority class (~5%). We want a model that is good at finding late orders.
- PR-AUC directly measures the trade-off between precision and recall for the positive class.
- ROC-AUC can be overly optimistic on imbalanced datasets because it includes the large number of True Negatives.


In [ ]:
champion_name  = comparison_df.iloc[0]['Model']
champion_model = models[champion_name]

print(f'Champion Model Selected : {champion_name}')
print(f'Val PR-AUC              : {comparison_df.iloc[0]["Val PR-AUC"]}')
print(f'Val ROC-AUC             : {comparison_df.iloc[0]["Val ROC-AUC"]}')

print('\nRanking Summary:')
for i, row in comparison_df.reset_index(drop=True).iterrows():
    marker = '  <-- CHAMPION' if row['Model'] == champion_name else ''
    print(f'  Rank {i+1}: {row["Model"]:<22} | PR-AUC: {row["Val PR-AUC"]}{marker}')


## 6. Decision Threshold Optimization
The default classification threshold of 0.5 is rarely optimal for imbalanced datasets.
We sweep thresholds from 0.05 to 0.90 on the **Validation Set** and select the threshold that maximizes F1-Score.

**Why not use X_test for this?** Threshold tuning on the test set would constitute data leakage — the model would be implicitly 'tuned' on test data. We only use X_val for this step.

In [ ]:
champion_val_probs = val_probs_dict[champion_name]

thresholds = np.linspace(0.05, 0.90, 171)
best_thresh, best_f1 = 0.5, 0.0
thresh_results = []

for t in thresholds:
    preds_t = (champion_val_probs >= t).astype(int)
    f1   = f1_score(y_val, preds_t, zero_division=0)
    rec  = recall_score(y_val, preds_t, zero_division=0)
    prec = precision_score(y_val, preds_t, zero_division=0)
    thresh_results.append({'threshold': round(t,3), 'f1': round(f1,4),
                            'recall': round(rec,4), 'precision': round(prec,4)})
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

best_thresh = round(best_thresh, 3)
thresh_df = pd.DataFrame(thresh_results)

print(f'Optimal Decision Threshold  : {best_thresh}')
print(f'Maximum Validation F1-Score : {best_f1:.4f}')
print(f'(Default threshold 0.5 F1   : {f1_score(y_val, (champion_val_probs>=0.5).astype(int)):.4f})')

print('\nTop 5 Threshold Candidates:')
print(thresh_df.sort_values("f1", ascending=False).head(5).to_string(index=False))


## 7. Champion Model — Validation Metrics at Optimal Threshold
We now evaluate the champion model at the optimal threshold on the Validation Set.

In [ ]:
val_preds_opt = (champion_val_probs >= best_thresh).astype(int)

print(f'=== {champion_name.upper()} — VALIDATION SET AT THRESHOLD {best_thresh} ===')
print(f'Val ROC-AUC  : {roc_auc_score(y_val, champion_val_probs):.4f}')
print(f'Val PR-AUC   : {average_precision_score(y_val, champion_val_probs):.4f}')
print(f'Val F1-Score : {f1_score(y_val, val_preds_opt):.4f}')
print(f'Val Recall   : {recall_score(y_val, val_preds_opt):.4f}')
print(f'Val Precision: {precision_score(y_val, val_preds_opt):.4f}')

print('\nClassification Report (Validation Set):')
print(classification_report(y_val, val_preds_opt, target_names=['On-Time (0)', 'Late (1)']))


## 8. FINAL TEST SET EVALUATION — Evaluated Once Only
> **IMPORTANT:** The test set (`X_test`) is evaluated **only once** here.
> It was never used during model training, hyperparameter tuning, or threshold selection.
> This ensures an unbiased estimate of real-world performance.

The champion model and the optimal threshold (selected on X_val) are now applied to X_test.

In [ ]:
test_probs = champion_model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_thresh).astype(int)

test_roc  = roc_auc_score(y_test, test_probs)
test_pr   = average_precision_score(y_test, test_probs)
test_f1   = f1_score(y_test, test_preds)
test_rec  = recall_score(y_test, test_preds)
test_prec = precision_score(y_test, test_preds)

print('=' * 60)
print('    FINAL TEST SET RESULTS — CHAMPION MODEL')
print('=' * 60)
print(f'  Champion Model    : {champion_name}')
print(f'  Threshold Used    : {best_thresh}')
print(f'  Test ROC-AUC      : {test_roc:.4f}')
print(f'  Test PR-AUC       : {test_pr:.4f}')
print(f'  Test F1-Score     : {test_f1:.4f}')
print(f'  Test Recall       : {test_rec:.4f}')
print(f'  Test Precision    : {test_prec:.4f}')
print('=' * 60)

print('\nClassification Report (Test Set):')
print(classification_report(y_test, test_preds, target_names=['On-Time (0)', 'Late (1)']))

cm = confusion_matrix(y_test, test_preds)
print('Confusion Matrix:')
print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}')
print(f'  FN={cm[1,0]:,}  TP={cm[1,1]:,}')
print(f'  Of {y_test.sum():,} actual late orders, our model correctly identified {cm[1,1]:,} ({cm[1,1]/y_test.sum()*100:.1f}%).')


## 9. Why Certain Models Perform Better or Worse
This section provides a principled analysis of the model performance differences.

In [ ]:
print('=== MODEL PERFORMANCE ANALYSIS ===')

print('1. MLP Neural Net (Champion):')
print('   - Captures non-linear feature interactions across all 67 features.')
print('   - StandardScaler normalization (already applied) allows MLP to converge efficiently.')
print('   - Early stopping prevents overfitting on the training set.')

print('\n2. Logistic Regression:')
print('   - Strong ROC-AUC due to linear separability of some features.')
print('   - Limited by its linear decision boundary — cannot capture non-linear patterns.')
print('   - class_weight=balanced helps with the class imbalance.')

print('\n3. LightGBM:')
print('   - Gradient boosting corrects errors of previous trees sequentially.')
print('   - scale_pos_weight directly upweights the minority (late) class.')
print('   - Lower PR-AUC than MLP possibly due to temporal overfitting in boosted trees.')

print('\n4. Random Forest:')
print('   - Bagging reduces variance but may underperform on rare class detection.')
print('   - balanced_subsample recalculates weights per bootstrap but can still miss rare patterns.')

print('\nGeneral Observation:')
print('   The gap between Val and Test metrics is consistent with temporal distribution shift.')
print('   Delivery patterns in later months (test period) differ from earlier months (train/val).')
print('   This is a known limitation of time-ordered splits and reflects real deployment challenges.')


## 10. Save Champion Model Artifact

In [ ]:
champion_artifact = {
    'model':         champion_model,
    'model_name':    champion_name,
    'threshold':     best_thresh,
    'val_roc_auc':   round(roc_auc_score(y_val, champion_val_probs), 4),
    'val_pr_auc':    round(average_precision_score(y_val, champion_val_probs), 4),
    'test_roc_auc':  round(test_roc, 4),
    'test_pr_auc':   round(test_pr, 4),
    'test_f1':       round(test_f1, 4),
    'test_recall':   round(test_rec, 4),
    'test_precision':round(test_prec, 4),
}

joblib.dump(champion_artifact, MODELS_DIR / 'champion_model.pkl')
print(f'Champion artifact saved: models/champion_model.pkl')
print(f'Comparison table saved : models/model_comparison.csv')
print(f'\nFinal Champion: {champion_name} | Threshold: {best_thresh}')
print(f'Test PR-AUC: {test_pr:.4f} | Test F1: {test_f1:.4f} | Test Recall: {test_rec:.4f}')
